In [ ]:
import os
import torch
import torch.nn as nn
import torch.fft
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Note: In a proper repository, you would import the model class from train.py 
# e.g., `from train import StateOfTheArtHybridDetector` instead of redefining it.
# It is redefined here so this script works entirely standalone.

class FrequencyConvLayer(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.amp_conv = nn.Conv2d(channels, channels, kernel_size=1)
        self.phase_conv = nn.Conv2d(channels, channels, kernel_size=1)
    def forward(self, x):
        x_f32 = x.float()
        freq_repr = torch.fft.rfft2(x_f32, norm="ortho")
        amplitude = torch.abs(freq_repr)
        phase = torch.angle(freq_repr)
        log_amplitude = torch.log(amplitude + 1e-8)
        amp_logits = self.amp_conv(log_amplitude.to(x.dtype))
        amp_mask = torch.sigmoid(amp_logits.float()) * 2.0
        learned_amp = amplitude * amp_mask
        phase_shift = self.phase_conv(phase.to(x.dtype))
        learned_phase = phase + phase_shift.float()
        complex_repr = learned_amp * torch.exp(1j * learned_phase)
        return torch.fft.irfft2(complex_repr, s=x.shape[-2:], norm="ortho").to(x.dtype)

class CrossModalityMixBlock(nn.Module):
    def __init__(self, spatial_channels, freq_channels):
        super().__init__()
        self.spatial_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(spatial_channels, freq_channels, kernel_size=1),
            nn.ReLU(), nn.Sigmoid()
        )
        self.freq_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(freq_channels, spatial_channels, kernel_size=1),
            nn.ReLU(), nn.Sigmoid()
        )
    def forward(self, spatial_feat, freq_feat):
        freq_gate = self.spatial_attention(spatial_feat)
        spatial_gate = self.freq_attention(freq_feat)
        return spatial_feat + (spatial_feat * spatial_gate), freq_feat + (freq_feat * freq_gate)

class StateOfTheArtHybridDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial_backbone = models.efficientnet_b3(weights=None) 
        self.spatial_features = self.spatial_backbone.features
        self.freq_stem = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=7, stride=2, padding=3, bias=False),
            nn.InstanceNorm2d(128, affine=True), nn.ReLU(), nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.freq_stream = FrequencyConvLayer(128)
        self.mix_block = CrossModalityMixBlock(spatial_channels=1536, freq_channels=128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(1536 + 128, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 1)
        )
    def forward(self, x):
        s_feat = self.spatial_features(x)
        f_feat = self.freq_stream(self.freq_stem(x))
        s_feat, f_feat = self.mix_block(s_feat, f_feat)
        s_vec = self.global_pool(s_feat).view(s_feat.size(0), -1)
        f_vec = self.global_pool(f_feat).view(f_feat.size(0), -1)
        return self.classifier(torch.cat((s_vec, f_vec), dim=1))

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = StateOfTheArtHybridDetector().to(device)

    weights_path = "models/model_v5_sota_efficientnet.pth"
    model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    model.eval()

    wild_deepfake_path = r"D:\AI_Datasets\wild deepfake\test"

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    wild_dataset = ImageFolder(root=wild_deepfake_path, transform=val_transform)
    wild_loader = DataLoader(wild_dataset, batch_size=32, shuffle=False, num_workers=0)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(wild_loader):
            images = images.to(device)
            with torch.amp.autocast('cuda'):
                logits = model(images)
                probs = torch.sigmoid(logits).cpu().numpy().flatten()
            
            all_preds.extend(probs)
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    binary_preds = (all_preds > 0.5).astype(int)

    final_accuracy = accuracy_score(all_labels, binary_preds) * 100
    print(f"WILD DEEPFAKE BLIND ACCURACY: {final_accuracy:.2f}%")
    print(classification_report(all_labels, binary_preds, target_names=wild_dataset.classes))

    plt.style.use('dark_background')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    cm = confusion_matrix(all_labels, binary_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='cool', cbar=False, 
                xticklabels=wild_dataset.classes, yticklabels=wild_dataset.classes, ax=ax1, annot_kws={"size": 16})
    ax1.set_title("Neural Decision Matrix (Cross-Domain)", fontsize=16, color='cyan', pad=15)
    ax1.set_xlabel("Predicted Class", fontsize=12)
    ax1.set_ylabel("Actual Class", fontsize=12)

    fpr, tpr, _ = roc_curve(all_labels, all_preds)
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, color='magenta', lw=3, label=f'Wild AUC = {roc_auc:.4f}')
    ax2.plot([0, 1], [0, 1], color='cyan', lw=2, linestyle='--')
    ax2.set_title("Receiver Operating Characteristic", fontsize=16, color='magenta', pad=15)
    ax2.set_xlabel("False Positive Rate", fontsize=12)
    ax2.set_ylabel("True Positive Rate", fontsize=12)
    ax2.legend(loc="lower right", fontsize=12)

    plt.tight_layout()
    plt.show()